In [2]:
from pathlib import Path
import pandas as pd

pasta_projeto = Path.cwd()
arquivo_tabela = pasta_projeto / "CD2022_Populacao_2010_Compatibilizada_20231222.csv"

if not arquivo_tabela.exists():
    raise FileNotFoundError(f"Tabela não encontrada em: {arquivo_tabela}")

arquivo_tabela

WindowsPath('h:/MATÉRIAS TECNICAS/TERCEIRA ETAPA/PYTHON/Atividade4/CD2022_Populacao_2010_Compatibilizada_20231222.csv')

In [3]:
dados_brutos = pd.read_excel(
    arquivo_tabela,
    sheet_name="Municípios",
    header=2,
    engine="openpyxl",
)

dados_brutos = dados_brutos.dropna(axis=1, how="all")
dados_brutos = dados_brutos[dados_brutos["UF"].astype(str).str.fullmatch(r"[A-Z]{2}")].copy()

populacao_municipios = dados_brutos.rename(columns={
    "COD. UF": "codigo_uf",
    "COD. MUNIC": "codigo_municipio",
    "NOME DO MUNICÍPIO": "municipio",
    "População 2010 (Alterações de Limites até 2022)1": "populacao_2010",
    "População Censo 2022": "populacao_2022",
})

populacao_municipios["codigo_uf"] = (
    populacao_municipios["codigo_uf"].astype(int).astype(str).str.zfill(2)
)
populacao_municipios["codigo_municipio"] = (
    populacao_municipios["codigo_municipio"].astype(int).astype(str).str.zfill(5)
)
populacao_municipios["codigo_ibge"] = (
    populacao_municipios["codigo_uf"] + populacao_municipios["codigo_municipio"]
)
populacao_municipios["populacao_2010"] = populacao_municipios["populacao_2010"].astype(int)
populacao_municipios["populacao_2022"] = populacao_municipios["populacao_2022"].astype(int)

populacao_municipios = populacao_municipios[[
    "UF", "codigo_uf", "codigo_municipio", "codigo_ibge",
    "municipio", "populacao_2010", "populacao_2022"
]].reset_index(drop=True)

print(f"Municípios encontrados: {len(populacao_municipios):,}".replace(",", "."))
print(populacao_municipios.head().to_string(index=False))

Municípios encontrados: 5.570
UF codigo_uf codigo_municipio codigo_ibge             municipio  populacao_2010  populacao_2022
RO        11            00015     1100015 Alta Floresta D'Oeste           24392           21494
RO        11            00023     1100023             Ariquemes           90353           96833
RO        11            00031     1100031                Cabixi            6313            5351
RO        11            00049     1100049                Cacoal           78574           86887
RO        11            00056     1100056            Cerejeiras           17029           15890


In [3]:
populacao_estados = (
    populacao_municipios.groupby("UF")[["populacao_2010", "populacao_2022"]]
    .sum()
    .reset_index()
)

populacao_estados["crescimento_absoluto"] = (
    populacao_estados["populacao_2022"] - populacao_estados["populacao_2010"]
)
populacao_estados["crescimento_percentual"] = (
    populacao_estados["crescimento_absoluto"]
    / populacao_estados["populacao_2010"]
    * 100
).round(2)

populacao_estados = populacao_estados.sort_values(
    by="crescimento_absoluto",
    ascending=False,
).reset_index(drop=True)

print(populacao_estados.to_string(index=False))

UF  populacao_2010  populacao_2022  crescimento_absoluto  crescimento_percentual
SP        41262199        44411238               3149039                    7.63
SC         6248436         7610361               1361925                   21.80
GO         6001789         7056495               1054706                   17.57
PR        10444526        11444380                999854                    9.57
MG        19597330        20539989                942659                    4.81
MT         3035122         3658649                623527                   20.54
PA         7581051         8120131                539080                    7.11
AM         3483985         3941613                457628                   13.14
CE         8451644         8794957                343313                    4.06
ES         3514952         3833712                318760                    9.07
MS         2449024         2757013                307989                   12.58
PE         8796055         9

In [4]:
arquivo_estados = pasta_projeto / "crescimento_populacao_por_estado.csv"

populacao_estados.to_csv(
    arquivo_estados,
    sep=";",
    index=False,
    encoding="utf-8-sig",
)

print(f"Arquivo salvo: {arquivo_estados.name}")

Arquivo salvo: crescimento_populacao_por_estado.csv


In [5]:
populacao_municipios["crescimento_absoluto"] = (
    populacao_municipios["populacao_2022"] - populacao_municipios["populacao_2010"]
)
populacao_municipios["crescimento_percentual"] = (
    populacao_municipios["crescimento_absoluto"]
    / populacao_municipios["populacao_2010"]
    * 100
).round(2)

populacao_municipios = populacao_municipios.sort_values(
    by="crescimento_absoluto",
    ascending=False,
).reset_index(drop=True)

print(populacao_municipios.head(20).to_string(index=False))

UF codigo_uf codigo_municipio codigo_ibge           municipio  populacao_2010  populacao_2022  crescimento_absoluto  crescimento_percentual
AM        13            02603     1302603              Manaus         1802014         2063689                261675                   14.52
DF        53            00108     5300108            Brasília         2572159         2817381                245222                    9.53
SP        35            50308     3550308           São Paulo        11253503        11451999                198496                    1.76
SP        35            52205     3552205            Sorocaba          586816          723682                136866                   23.32
GO        52            08707     5208707             Goiânia         1301912         1437366                135454                   10.40
RR        14            00100     1400100           Boa Vista          284313          413486                129173                   45.43
SC        42        

In [6]:
arquivo_municipios = pasta_projeto / "crescimento_populacao_por_municipio.csv"

populacao_municipios.to_csv(
    arquivo_municipios,
    sep=";",
    index=False,
    encoding="utf-8-sig",
)

print(f"Arquivo salvo: {arquivo_municipios.name}")

Arquivo salvo: crescimento_populacao_por_municipio.csv
